# Connection Check & Catalog Discovery

This notebook:
1. Confirms AWS credentials work.
2. Lists the Glue databases and tables (so we see the *current* names).
3. Pulls a tiny sample from the dimension tables.
4. Locates the live telemetry (fact) table and peeks at the compliance results.

**Before running:** make sure you have logged in and selected the profile:

```powershell
aws sso login --profile ciccada
$env:AWS_PROFILE = "ciccada"
```

## 1. Credentials check

In [ ]:
import boto3
session = boto3.Session(profile_name="ciccada", region_name="ap-southeast-2")
ident = session.client("sts").get_caller_identity()
print("Account:", ident["Account"])
print("Identity:", ident["Arn"])
# If this errors with 'Unable to locate credentials', run `aws sso login` and
# set AWS_PROFILE, then restart the kernel.

## 2. The catalog
Mapping (`SolA_ts4`, `SolA_circuits`) to whatever they are called today.

In [ ]:
from aws_config import *
databases()

In [ ]:
# Tables in the main analytics database
tables("solar_analytics")[["Database", "Table", "TableType"]]

In [ ]:
# The Iceberg database
tables("solar_analytics_iceberg")[["Database", "Table", "TableType"]]

## 3. Sample the dimension tables

In [ ]:
aq("SELECT * FROM circuits LIMIT 5")

In [ ]:
aq("SELECT * FROM sites LIMIT 5")

In [ ]:
# The partition lookup tells you which (year, month) partitions actually exist
aq("SELECT * FROM partition_lookup LIMIT 20")

## 4. Find the live telemetry table

In [ ]:
# partition-pruned sample
YEAR = 2025
MONTH = 1
sample = aq(f'''
    SELECT circuit_id, t_stamp, voltage, power, energy_reactive
    FROM {FACT_TABLE}
    WHERE is_pv = True AND year = {YEAR} AND month = {MONTH}
      AND circuit_id = 547781
    ORDER BY t_stamp
    LIMIT 20
''', database=FACT_DB)
sample

## 5. Analysis outputs

These tables are the *results* the report was built from. Their schemas tell us
exactly what the `SolA2024_Analysis` notebooks ultimately produced.

In [ ]:
aq("SELECT * FROM compliance_voltwatt LIMIT 5")

In [ ]:
# aq("SELECT * FROM compliance_voltvar LIMIT 5")

In [ ]:
# What inverter metadata was inferred (nameplate capacity, etc.)
aq("SELECT * FROM meta_single_inverters LIMIT 5")